<a href="https://colab.research.google.com/github/niranjan-1785/ml_lab/blob/main/exp_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

In [11]:
categories = [
    'alt.atheism',
    'soc.religion.christian',
    'comp.graphics',
    'sci.med'
]
print("Loading 20 Newsgroups dataset...")

train_data = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)
test_data = fetch_20newsgroups(
    subset='test',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)
print("Dataset loaded successfully.")
print("Training samples:", len(train_data.data))
print("Testing samples:", len(test_data.data))

Loading 20 Newsgroups dataset...
Dataset loaded successfully.
Training samples: 2257
Testing samples: 1502


In [12]:
vectorizer = CountVectorizer(
    stop_words='english',
    max_features=5000
)
X_train = vectorizer.fit_transform(train_data.data).toarray()
y_train = train_data.target
X_test = vectorizer.transform(test_data.data).toarray()
y_test = test_data.target
n_classes = len(categories)
n_features = X_train.shape[1]
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Number of classes:", n_classes)
print("Number of features:", n_features)

Train shape: (2257, 5000)
Test shape: (1502, 5000)
Number of classes: 4
Number of features: 5000


In [13]:
class_counts = np.bincount(y_train)
p_labels = class_counts / len(y_train)
print("Class counts:", class_counts)
print("Class priors:", p_labels)

N_c = np.zeros((n_classes, n_features))
for c in range(n_classes):
    N_c[c, :] = X_train[y_train == c].sum(axis=0)
print("Word counts aggregated successfully.")
print("N_c shape:", N_c.shape)

Class counts: [480 584 594 599]
Class priors: [0.21267169 0.25875055 0.26318121 0.26539654]
Word counts aggregated successfully.
N_c shape: (4, 5000)


In [14]:
def predict_naive_bayes(X, theta, p_labels):
    """
    Predict classes using log-probabilities.
    X: Document-word count matrix
    theta: Word probabilities for each class
    p_labels: Class prior probabilities
    """
    with np.errstate(divide='ignore'):
        log_theta = np.log(theta)
    log_theta[np.isinf(log_theta)] = -1e9
    log_posterior = X @ log_theta.T + np.log(p_labels)
    return np.argmax(log_posterior, axis=1)
print("Prediction function defined successfully.")

Prediction function defined successfully.


In [15]:
theta_mle = N_c / N_c.sum(axis=1, keepdims=True)
y_pred_mle = predict_naive_bayes(
    X_test,
    theta_mle,
    p_labels
)
mle_accuracy = accuracy_score(y_test, y_pred_mle)
print(f"[MLE] Test Accuracy: {mle_accuracy:.4f}")

[MLE] Test Accuracy: 0.7710


In [16]:
priors_to_test = {
    "Lidstone Smoothing (Alpha = 1.01)": np.ones(n_features) * 1.01,
    "Laplace Smoothing (Alpha = 2.0)": np.ones(n_features) * 2.0,
    "Strong Dirichlet Prior (Alpha = 10.0)": np.ones(n_features) * 10.0,
    "Non-Uniform Prior (Empirical Basis)": (
        1.0 + (X_train.sum(axis=0) / X_train.sum() * 100)
    )
}
print("Prior distributions defined successfully.")
print("Number of priors to test:", len(priors_to_test))

Prior distributions defined successfully.
Number of priors to test: 4


In [17]:
print("Evaluating MAP Predictions:")
print("-" * 65)
for name, alpha in priors_to_test.items():
    numerator = N_c + (alpha - 1)
    denominator = numerator.sum(axis=1, keepdims=True)
    theta_map = numerator / denominator
    y_pred_map = predict_naive_bayes(
        X_test,
        theta_map,
        p_labels
    )
    map_accuracy = accuracy_score(y_test, y_pred_map)
    print(f"{name:38} | Test Accuracy: {map_accuracy:.4f}")

Evaluating MAP Predictions:
-----------------------------------------------------------------
Lidstone Smoothing (Alpha = 1.01)      | Test Accuracy: 0.8103
Laplace Smoothing (Alpha = 2.0)        | Test Accuracy: 0.8182
Strong Dirichlet Prior (Alpha = 10.0)  | Test Accuracy: 0.7863
Non-Uniform Prior (Empirical Basis)    | Test Accuracy: 0.8063


In [18]:
results = {
    "MLE": mle_accuracy
}
for name, alpha in priors_to_test.items():
    numerator = N_c + (alpha - 1)
    denominator = numerator.sum(axis=1, keepdims=True)
    theta_map = numerator / denominator
    y_pred_map = predict_naive_bayes(
        X_test,
        theta_map,
        p_labels
    )
    results[name] = accuracy_score(y_test, y_pred_map)
print("\nFinal Accuracy Comparison")
print("-" * 65)
for method, accuracy in results.items():
    print(f"{method:45} : {accuracy:.4f}")


Final Accuracy Comparison
-----------------------------------------------------------------
MLE                                           : 0.7710
Lidstone Smoothing (Alpha = 1.01)             : 0.8103
Laplace Smoothing (Alpha = 2.0)               : 0.8182
Strong Dirichlet Prior (Alpha = 10.0)         : 0.7863
Non-Uniform Prior (Empirical Basis)           : 0.8063
